# Dashboard de Mantenimiento — BoeIoT

**Audiencia:** equipo de mantenimiento (diagnóstico de motores, detección de anomalías, identificación de vuelos sospechosos).

**Fuente:** capa Gold — tabla `flight_summary.parquet` (1 fila por vuelo, ~35 KPIs). Ver [`docs/gold_to_dashboard.md`](../../docs/gold_to_dashboard.md) para el esquema completo.

Cada sección consume columnas específicas del Gold y muestra una visión accionable de la flota.

In [ ]:
import io
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:4566',
    aws_access_key_id='test',
    aws_secret_access_key='test',
    region_name='us-east-1',
)

GOLD_BUCKET = 'dos-boeing-737-max-gold-layer'
GOLD_KEY    = 'flight_summary.parquet'

response = s3.get_object(Bucket=GOLD_BUCKET, Key=GOLD_KEY)
df = pd.read_parquet(io.BytesIO(response['Body'].read()))
print(f'Loaded {len(df)} flights, {df.shape[1]} KPIs from Gold.')
df.head()

## 1. Resumen de flota

Vista de un vistazo: cuántos vuelos hay, cuántas horas se han operado, salud media y volumen de anomalías. Las cifras en rojo marcan vuelos que requieren inspección.

In [ ]:
total_flights = len(df)
total_hours = df['flight_duration_min'].sum() / 60
avg_health = df['engine_health_score'].mean()
total_anomalies = int(df['total_anomaly_count'].sum())
critical_flights = int((df['engine_health_score'] < 60).sum())

summary = pd.DataFrame({
    'Indicador': [
        'Vuelos analizados',
        'Horas totales de operación',
        'Health score promedio',
        'Anomalías totales detectadas',
        'Vuelos con score < 60 (alerta)'
    ],
    'Valor': [
        f'{total_flights:,}',
        f'{total_hours:,.1f} h',
        f'{avg_health:.1f} / 100',
        f'{total_anomalies:,}',
        f'{critical_flights}'
    ]
})
summary.style.hide(axis='index').set_properties(**{'font-size': '13pt'})

## 2. Top 10 vuelos críticos

Vuelos con el peor `engine_health_score`. Estos son candidatos prioritarios para inspección.  
Score < 50 (rojo) indica problemas serios; score 50–70 (naranja) requiere seguimiento.

In [ ]:
top10 = df.nsmallest(10, 'engine_health_score')[
    ['flight_id', 'engine_health_score', 'total_anomaly_count', 'max_cht_spread', 'max_egt_spread']
].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#c0392b' if s < 50 else '#e67e22' if s < 70 else '#27ae60' for s in top10['engine_health_score']]
ax.barh(top10['flight_id'].astype(str), top10['engine_health_score'], color=colors)
ax.invert_yaxis()
ax.set_xlabel('Engine health score (0-100)')
ax.set_title('Top 10 vuelos críticos (menor score primero)')
ax.axvline(50, color='#c0392b', linestyle='--', alpha=0.4, label='Crítico (< 50)')
ax.axvline(70, color='#e67e22', linestyle='--', alpha=0.4, label='Atención (< 70)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

top10

## 3. Distribución del Health Score

Distribución de la salud del motor en toda la flota. Las líneas verticales marcan los percentiles 10 y 25, definiendo umbrales de **crítico** y **atención** respectivamente.

In [ ]:
p10 = df['engine_health_score'].quantile(0.10)
p25 = df['engine_health_score'].quantile(0.25)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(df['engine_health_score'], bins=20, color='#3498db', edgecolor='white')
ax.axvline(p10, color='#c0392b', linestyle='--', label=f'p10 (crítico) = {p10:.1f}')
ax.axvline(p25, color='#e67e22', linestyle='--', label=f'p25 (atención) = {p25:.1f}')
ax.set_xlabel('Engine health score')
ax.set_ylabel('# vuelos')
ax.set_title('Distribución del Health Score en la flota')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Mapa de calor de anomalías

Muestra los **top 30 vuelos** con más anomalías y la distribución de las mismas por tipo de evento. Permite ver patrones: ¿hay vuelos con problemas concentrados en un solo subsistema, o son problemas múltiples simultáneos?

In [ ]:
top30 = df.nlargest(30, 'total_anomaly_count')
heatmap_data = top30.set_index('flight_id')[
    ['cht_imbalance_events', 'egt_imbalance_events', 'low_oil_pressure_events', 'high_oil_temp_events']
]
heatmap_data.columns = ['CHT imbalance', 'EGT imbalance', 'Low oil pressure', 'High oil temp']

fig, ax = plt.subplots(figsize=(7, 9))
sns.heatmap(heatmap_data, cmap='Reds', annot=False, cbar_kws={'label': '# eventos'}, ax=ax)
ax.set_title('Anomalías por vuelo y tipo de evento (top 30)')
ax.set_xlabel('Tipo de evento')
ax.set_ylabel('Flight ID')
plt.tight_layout()
plt.show()

## 5. Presión vs Temperatura de aceite

Cuadrante de riesgo: alta temperatura combinada con baja presión indica **falla inminente de lubricación**. El tamaño del punto refleja el conteo total de anomalías; el color, el health score (rojo = malo, verde = bueno).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    df['avg_oil_temp'], df['min_oil_pressure'],
    s=20 + df['total_anomaly_count'] * 3,
    c=df['engine_health_score'], cmap='RdYlGn',
    alpha=0.75, edgecolor='black', linewidth=0.4,
)
ax.axvspan(df['avg_oil_temp'].quantile(0.75), df['avg_oil_temp'].max(),
           ymax=(df['min_oil_pressure'].quantile(0.25) - df['min_oil_pressure'].min()) /
                (df['min_oil_pressure'].max() - df['min_oil_pressure'].min()),
           color='red', alpha=0.08, label='Cuadrante peligroso')
ax.set_xlabel('Temperatura promedio de aceite (normalizada)')
ax.set_ylabel('Presión mínima de aceite (normalizada)')
ax.set_title('Riesgo de lubricación — Presión vs Temperatura de aceite')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Engine health score')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 6. Desbalance entre cilindros (CHT y EGT)

Spread = `max - min` entre los 4 cilindros del motor. Un spread alto indica que un cilindro opera fuera de sincronía (posible misfire, fuga de compresión, o falla de inyector). Se segmenta por la **fase dominante** del vuelo para identificar si el problema aparece en alguna fase específica.

In [ ]:
df['dominant_phase'] = df[['pct_ascent', 'pct_cruise', 'pct_descent']].idxmax(axis=1).str.replace('pct_', '')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
sns.boxplot(data=df, x='dominant_phase', y='max_cht_spread',
            order=['ascent', 'cruise', 'descent'], ax=axes[0], palette='Set2')
axes[0].set_title('Máximo CHT spread por fase dominante')
axes[0].set_xlabel('Fase dominante del vuelo')
axes[0].set_ylabel('Max CHT spread (normalizado)')

sns.boxplot(data=df, x='dominant_phase', y='max_egt_spread',
            order=['ascent', 'cruise', 'descent'], ax=axes[1], palette='Set2')
axes[1].set_title('Máximo EGT spread por fase dominante')
axes[1].set_xlabel('Fase dominante del vuelo')
axes[1].set_ylabel('Max EGT spread (normalizado)')

plt.tight_layout()
plt.show()

## 7. Consumo de combustible

Vuelos ordenados por combustible consumido. Los vuelos con **alto desbalance entre tanques** (`max_fuel_imbalance` por encima del p75) se resaltan en naranja — esto puede indicar fuga, sensor defectuoso o transferencia asimétrica.

In [ ]:
top_fuel = df.nlargest(20, 'fuel_consumed').sort_values('fuel_consumed', ascending=True)
imbalance_threshold = df['max_fuel_imbalance'].quantile(0.75)
colors = ['#e67e22' if x > imbalance_threshold else '#3498db' for x in top_fuel['max_fuel_imbalance']]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_fuel['flight_id'].astype(str), top_fuel['fuel_consumed'], color=colors)
ax.set_xlabel('Combustible consumido (normalizado)')
ax.set_title('Top 20 vuelos por consumo de combustible')
ax.text(0.99, 0.02,
        f'Naranja = desbalance > p75 ({imbalance_threshold:.3f})',
        transform=ax.transAxes, ha='right', fontsize=9,
        bbox=dict(facecolor='white', alpha=0.8))
plt.tight_layout()
plt.show()

## 8. Calidad de telemetría vs. Health score

Cruza la **disponibilidad de sensores** con el health score. Si hay correlación negativa fuerte (vuelos con datos degradados tienen scores bajos), parte del score puede ser un **falso positivo** causado por datos faltantes, no por problemas reales del motor.

Si los puntos están dispersos sin correlación clara, los problemas detectados son reales (independientes de la calidad del dato).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sc = ax.scatter(
    df['sensor_availability'], df['engine_health_score'],
    c=df['total_anomaly_count'], cmap='viridis', s=40, alpha=0.7, edgecolor='black', linewidth=0.4,
)
ax.set_xlabel('Sensor availability (% lecturas válidas)')
ax.set_ylabel('Engine health score')
ax.set_title('Calidad de telemetría vs Salud del motor')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Total anomaly count')

corr = df[['sensor_availability', 'engine_health_score']].corr().iloc[0, 1]
ax.text(0.02, 0.98,
        f'Pearson r = {corr:+.3f}',
        transform=ax.transAxes, va='top',
        bbox=dict(facecolor='white', alpha=0.85))
plt.tight_layout()
plt.show()

---

## Cómo extender este dashboard

1. Agregar un nuevo KPI en `src/scripts/silver_to_gold_etl.py` (añadir entrada al `groupby().agg()` o cálculo derivado).
2. Re-ejecutar el pipeline (`uv run src/scripts/silver_to_gold_etl.py`).
3. Crear una nueva celda markdown + code aquí siguiendo el patrón de las secciones anteriores.
4. Documentar el nuevo KPI y su visualización en [`docs/gold_to_dashboard.md`](../../docs/gold_to_dashboard.md).